<a href="https://colab.research.google.com/github/jayatigupta05/VEP/blob/main/GenoHybrid_VEP_1_Data_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 0: Preparing Data

## Processing raw data

### Colab setup and project folders

In [13]:
from pathlib import Path

from google.colab import drive
import os

if not os.path.exists("/content/drive/MyDrive"):
    print("Google Drive not mounted. Mounting...")
    drive.mount("/content/drive")
else:
    print("Google Drive already mounted.")

REFERENCE_DIR = Path("/content/drive/MyDrive/variant_effect_prediction")
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

Google Drive already mounted.


In [ ]:
if not os.path.exists("/content/drive/MyDrive/variant_effect_prediction/data/references/hg38.phyloP100way.bw"):
    print("Downloading ..... ")
    !wget -c -O "/content/drive/MyDrive/variant_effect_prediction/data/references/hg38.phyloP100way.bw" \ "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/phyloP100way/hg38.phyloP100way.bw"
else:
    print("Exists")

Exists


In [ ]:
if not os.path.exists("/content/drive/MyDrive/variant_effect_prediction/data/references/hg38.phastCons100way.bw"):
    print("Downloading ..... ")
    !wget -c -O "/content/drive/MyDrive/variant_effect_prediction/data/references/hg38.phastCons100way.bw" \ "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/phastCons100way/hg38.phastCons100way.bw"
else:
    print("Exists")

Exists


In [ ]:
!pip -q install pyarrow py2bit pyBigWig tqdm

import os
import gc
import gzip
import glob
import json
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
PROJECT_PATH = Path("/content/drive/MyDrive/variant_effect_prediction")
RAW_DIR = PROJECT_PATH / "data" / "raw"
PROCESSED_DIR = PROJECT_PATH / "data" / "processed"
REFERENCE_DIR = PROJECT_PATH / "data" / "references"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_PATH)

print(f"Current working directory: {os.getcwd()}")
print(f"Raw data:       {RAW_DIR}")
print(f"Processed data: {PROCESSED_DIR}")
print(f"Reference data: {REFERENCE_DIR}")

Current working directory: /content/drive/MyDrive/variant_effect_prediction
Raw data:       /content/drive/MyDrive/variant_effect_prediction/data/raw
Processed data: /content/drive/MyDrive/variant_effect_prediction/data/processed
Reference data: /content/drive/MyDrive/variant_effect_prediction/data/references


In [ ]:
SEED = 42
SAMPLE_SIZE = 00_000

CHUNK_SIZE = 100_000
REUSE_EXISTING_CHUNKS = True

np.random.seed(SEED)

print("Sample size:", SAMPLE_SIZE)
print("Seed:", SEED)

Sample size: 100000
Seed: 42


In [ ]:
VARIANT_SUMMARY = RAW_DIR / "variant_summary.txt"
CLINVAR_VCF = RAW_DIR / "clinvar_grch38.vcf.gz"

HG38_2BIT = REFERENCE_DIR / "hg38.2bit"
PHYLOP_BW = REFERENCE_DIR / "hg38.phyloP100way.bw"
PHASTCONS_BW = REFERENCE_DIR / "hg38.phastCons100way.bw"

URLS = {
    "variant_summary_gz":
        "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz",

    "clinvar_vcf":
        "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz",

    "hg38_2bit":
        "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/latest/hg38.2bit",

    "phylop":
        "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/phyloP100way/hg38.phyloP100way.bw",

    "phastcons":
        "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/phastCons100way/hg38.phastCons100way.bw",
}


def download_if_missing(url, path):
    path = Path(path)

    if path.exists() and path.stat().st_size > 0:
        print(f"Exists: {path.name} ({path.stat().st_size / 1e6:.1f} MB)")
        return

    print(f"Downloading: {path.name}")
    subprocess.run(
        ["wget", "-c", "-O", str(path), url],
        check=True
    )


# variant_summary.txt is distributed gzipped.
if not VARIANT_SUMMARY.exists():
    gz_path = RAW_DIR / "variant_summary.txt.gz"
    download_if_missing(URLS["variant_summary_gz"], gz_path)

    print("Decompressing variant_summary.txt...")
    with gzip.open(gz_path, "rb") as src, open(VARIANT_SUMMARY, "wb") as dst:
        shutil.copyfileobj(src, dst)

    print("Saved:", VARIANT_SUMMARY)
else:
    print("Exists:", VARIANT_SUMMARY)

download_if_missing(URLS["clinvar_vcf"], CLINVAR_VCF)
download_if_missing(URLS["hg38_2bit"], HG38_2BIT)
download_if_missing(URLS["phylop"], PHYLOP_BW)
download_if_missing(URLS["phastcons"], PHASTCONS_BW)

Exists: /content/drive/MyDrive/variant_effect_prediction/data/raw/variant_summary.txt
Exists: clinvar_grch38.vcf.gz (200.1 MB)
Exists: hg38.2bit (858.8 MB)
Exists: hg38.phyloP100way.bw (9870.1 MB)
Exists: hg38.phastCons100way.bw (5886.4 MB)


In [ ]:
!ls -lh "/content/drive/MyDrive/variant_effect_prediction/data/references"

total 16G
-rw------- 1 root root 820M Oct 25  2022 hg38.2bit
-rw------- 1 root root 5.5G May  8  2015 hg38.phastCons100way.bw
-rw------- 1 root root 9.2G May  8  2015 hg38.phyloP100way.bw


### Loading raw data

In [ ]:
RAW_FILE_PATH = str(VARIANT_SUMMARY)

with open(RAW_FILE_PATH, 'r') as f:
    for _ in range(5):
        print(f.readline().strip())

#AlleleID	Type	Name	GeneID	GeneSymbol	HGNC_ID	ClinicalSignificance	ClinSigSimple	LastEvaluated	RS# (dbSNP)	nsv/esv (dbVar)	RCVaccession	PhenotypeIDS	PhenotypeList	Origin	OriginSimple	Assembly	ChromosomeAccession	Chromosome	Start	Stop	ReferenceAllele	AlternateAllele	Cytogenetic	ReviewStatus	NumberSubmitters	Guidelines	TestedInGTR	OtherIDs	SubmitterCategories	VariationID	PositionVCF	ReferenceAlleleVCF	AlternateAlleleVCF	SomaticClinicalImpact	SomaticClinicalImpactLastEvaluated	ReviewStatusClinicalImpact	Oncogenicity	OncogenicityLastEvaluated	ReviewStatusOncogenicity	SCVsForAggregateGermlineClassification	SCVsForAggregateSomaticClinicalImpact	SCVsForAggregateOncogenicityClassification
15041	Indel	NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTAACTGTAAA (p.Arg27_Ile28delinsLeuLeuTer)	9907	AP5Z1	HGNC:22197	Pathogenic/Likely pathogenic	1	Dec 17, 2024	397704705	-	RCV000000012|RCV005255549|RCV004998069	MONDO:MONDO:0013342,MedGen:C3150901,OMIM:613647,Orphanet:306511||MedGen:C3661900	Hereditary spa

### Filtering and chunking

In [ ]:
cols_to_keep = [
    'Type',
    'GeneSymbol',
    'ClinicalSignificance',
    'Assembly',
    'Chromosome',
    'Start',
    'Stop',
    'ReferenceAlleleVCF',
    'AlternateAlleleVCF',
    'NumberSubmitters',
    'OriginSimple',
    'ReviewStatus',
    'SomaticClinicalImpact'
]

csv_reader = pd.read_csv(
    RAW_FILE_PATH,
    sep='\t',
    chunksize=CHUNK_SIZE,
    low_memory=False
)

print("Processing started...")

written = 0

for index, chunk in enumerate(csv_reader):

    output_filename = PROCESSED_DIR / f"clinvar_clean_part_{index}.parquet"

    # if REUSE_EXISTING_CHUNKS and output_filename.exists():
    #     print(f"-> Part {index} already exists. Skipping.")
    #     written += 1
    #     continue

    if '#AlleleID' in chunk.columns:
        chunk = chunk.rename(columns={'#AlleleID': 'AlleleID'})

    # Keep only GRCh38.
    if 'Assembly' in chunk.columns:
        chunk = chunk[chunk['Assembly'] == 'GRCh38']

    if chunk.empty:
        continue

    keep = [c for c in cols_to_keep if c in chunk.columns]
    clean_chunk = chunk[keep].copy()

    clean_chunk = clean_chunk.dropna(
        subset=[
            'ClinicalSignificance',
            'ReferenceAlleleVCF',
            'AlternateAlleleVCF',
            'Chromosome',
            'Start'
        ]
    )

    # Normalise alleles now; strict SNV filtering is done later.
    clean_chunk['ReferenceAlleleVCF'] = (
        clean_chunk['ReferenceAlleleVCF']
        .astype(str)
        .str.upper()
    )

    clean_chunk['AlternateAlleleVCF'] = (
        clean_chunk['AlternateAlleleVCF']
        .astype(str)
        .str.upper()
    )

    # Legacy REF / ALT one-hot columns retained for comparison
    # with the prototype model.
    for base in ['A', 'C', 'G', 'T']:
        clean_chunk[f'Ref_{base}'] = (
            clean_chunk['ReferenceAlleleVCF'] == base
        ).astype(np.float32)

        clean_chunk[f'Alt_{base}'] = (
            clean_chunk['AlternateAlleleVCF'] == base
        ).astype(np.float32)

    clean_chunk['Tab_Scaled_Submitters'] = np.log1p(
        pd.to_numeric(
            clean_chunk.get('NumberSubmitters', 0),
            errors='coerce'
        ).fillna(0)
    ).astype(np.float32)

    # Preserve raw origin and also create the original prototype code.
    clean_chunk['Tab_Origin_Numeric'] = (
        clean_chunk.get('OriginSimple', '')
        .astype(str)
        .str.lower()
        .apply(
            lambda x:
                1.0 if 'germline' in x
                else (2.0 if 'somatic' in x else 0.0)
        )
        .astype(np.float32)
    )

    def map_review_to_stars(status_str):
        status = str(status_str).lower()

        if 'practice guideline' in status:
            return 4.0

        if 'reviewed by expert panel' in status:
            return 3.0

        if 'criteria provided, multiple submitters' in status:
            return 2.0

        if 'criteria provided, single submitter' in status:
            return 1.0

        return 0.0

    clean_chunk['Tab_Review_Score'] = (
        clean_chunk.get('ReviewStatus', '')
        .apply(map_review_to_stars)
        .astype(np.float32)
    )

    clean_chunk['Tab_Somatic_Impact'] = (
        clean_chunk.get('SomaticClinicalImpact', pd.Series(index=clean_chunk.index))
        .notna()
        .astype(np.float32)
    )

    if clean_chunk.empty:
        continue

    clean_chunk.to_parquet(
        output_filename,
        index=False
    )

    written += 1
    print(f"-> Successfully processed Part {index}")

    del chunk, clean_chunk
    gc.collect()

print(f"\nAll done! Chunk files available: {written}")

Processing started...
-> Successfully processed Part 0
-> Successfully processed Part 1
-> Successfully processed Part 2
-> Successfully processed Part 3
-> Successfully processed Part 4
-> Successfully processed Part 5
-> Successfully processed Part 6
-> Successfully processed Part 7
-> Successfully processed Part 8
-> Successfully processed Part 9
-> Successfully processed Part 10
-> Successfully processed Part 11
-> Successfully processed Part 12
-> Successfully processed Part 13
-> Successfully processed Part 14
-> Successfully processed Part 15
-> Successfully processed Part 16
-> Successfully processed Part 17
-> Successfully processed Part 18
-> Successfully processed Part 19
-> Successfully processed Part 20
-> Successfully processed Part 21
-> Successfully processed Part 22
-> Successfully processed Part 23
-> Successfully processed Part 24
-> Successfully processed Part 25
-> Successfully processed Part 26
-> Successfully processed Part 27
-> Successfully processed Part 28
->

### Standardizing Data

- Map ClinVar labels to the three project classes
- Keep standard single-base substitutions only
- Remove exact duplicate variants
- Preserve the original REF/ALT features for backward compatibility

In [14]:
processed_files = sorted(
    glob.glob(
        str(PROCESSED_DIR / 'clinvar_clean_part_*.parquet')
    )
)

print("Processed files:", len(processed_files))

label_mapping = {
    # Pathogenic -> Class 1
    'Pathogenic': 1,
    'Pathogenic/Likely pathogenic': 1,
    'Likely pathogenic': 1,

    # Benign -> Class 0
    'Benign': 0,
    'Benign/Likely benign': 0,
    'Likely benign': 0,

    # VUS / uncertain / conflicting -> Class 2
    'Uncertain significance': 2,
    'Conflicting interpretations of pathogenicity': 2,
    'Uncertain significance, risk factor': 2,
    'Conflicting interpretations of pathogenicity, risk factor': 2
}

all_chunks = []

for file_path in tqdm(
    processed_files,
    desc="Standardizing ClinVar chunks"
):
    df = pd.read_parquet(file_path)

    df['Label'] = df['ClinicalSignificance'].map(label_mapping)

    df = df.dropna(subset=['Label'])
    df['Label'] = df['Label'].astype(np.int8)

    df['ReferenceAlleleVCF'] = (
        df['ReferenceAlleleVCF']
        .astype(str)
        .str.upper()
    )

    df['AlternateAlleleVCF'] = (
        df['AlternateAlleleVCF']
        .astype(str)
        .str.upper()
    )

    # Exact SNVs only.
    df = df[
        (df['ReferenceAlleleVCF'].str.len() == 1) &
        (df['AlternateAlleleVCF'].str.len() == 1)
    ]

    all_chunks.append(df)

master_df = pd.concat(
    all_chunks,
    axis=0,
    ignore_index=True
)

del all_chunks
gc.collect()

standard_bases = {'A', 'C', 'G', 'T'}

master_df = master_df[
    master_df['ReferenceAlleleVCF'].isin(standard_bases) &
    master_df['AlternateAlleleVCF'].isin(standard_bases)
].copy()

master_df['Chromosome'] = (
    master_df['Chromosome']
    .astype(str)
    .str.replace('chr', '', regex=False)
)

valid_chroms = {str(i) for i in range(1, 23)} | {'X', 'Y'}

master_df = master_df[
    master_df['Chromosome'].isin(valid_chroms)
].copy()

master_df['Start'] = pd.to_numeric(
    master_df['Start'],
    errors='coerce'
)

master_df = master_df.dropna(subset=['Start'])
master_df['Start'] = master_df['Start'].astype(np.int64)

# Remove exact duplicate variant records.
master_df = master_df.drop_duplicates(
    subset=[
        'Chromosome',
        'Start',
        'ReferenceAlleleVCF',
        'AlternateAlleleVCF',
        'Label'
    ]
).reset_index(drop=True)

print("Total valid labelled SNVs:", f"{len(master_df):,}")
print(master_df['Label'].value_counts().sort_index())
print((master_df['Label'].value_counts(normalize=True).sort_index() * 100).round(2))

Processed files: 90


Standardizing ClinVar chunks:   0%|          | 0/90 [00:00<?, ?it/s]

Total valid labelled SNVs: 3,726,707
Label
0    1288701
1     187070
2    2250936
Name: count, dtype: int64
Label
0    34.58
1     5.02
2    60.40
Name: proportion, dtype: float64


### Sampling

Expensive sequence and conservation annotation happens **after sampling**.\

In [15]:
if SAMPLE_SIZE is not None and len(master_df) > SAMPLE_SIZE:

    sampled_df = master_df.sample(
        n=SAMPLE_SIZE,
        random_state=SEED
    ).reset_index(drop=True)

else:
    sampled_df = master_df.reset_index(drop=True).copy()

# Permanent graph node ID for this exact sampled panel.
sampled_df['node_id'] = np.arange(
    len(sampled_df),
    dtype=np.int64
)

print("Working sample:", f"{len(sampled_df):,}")
print(sampled_df['Label'].value_counts().sort_index())
print((sampled_df['Label'].value_counts(normalize=True).sort_index() * 100).round(2))

Working sample: 100,000
Label
0    34621
1     5135
2    60244
Name: count, dtype: int64
Label
0    34.62
1     5.14
2    60.24
Name: proportion, dtype: float64


In [16]:
# Free the full ~million-row dataframe before genomic annotation.
# Reload it later from chunk files if it is ever needed again.

del master_df
gc.collect()

print("Memory released. Continuing with sampled_df only.")

Memory released. Continuing with sampled_df only.


# Step 1: Add Genomic Sequence and Conservation

In [17]:
LOCAL_REF = Path("/content/genohybrid_reference")
LOCAL_REF.mkdir(parents=True, exist_ok=True)

LOCAL_2BIT = LOCAL_REF / HG38_2BIT.name
LOCAL_PHYLOP = LOCAL_REF / PHYLOP_BW.name
LOCAL_PHASTCONS = LOCAL_REF / PHASTCONS_BW.name


def copy_if_missing(src, dst):
    src = Path(src)
    dst = Path(dst)

    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        print("Local cache exists:", dst.name)
        return

    print("Copying to Colab local disk:", src.name)
    shutil.copy2(src, dst)


copy_if_missing(HG38_2BIT, LOCAL_2BIT)
copy_if_missing(PHYLOP_BW, LOCAL_PHYLOP)
copy_if_missing(PHASTCONS_BW, LOCAL_PHASTCONS)

Copying to Colab local disk: hg38.2bit
Copying to Colab local disk: hg38.phyloP100way.bw
Copying to Colab local disk: hg38.phastCons100way.bw


## Extract 201 bp GRCh38 sequence windows + PhyloP + PhastCons

We store **201 bp** around each variant (`100 bp + variant + 100 bp`).

Notebook 2 can then crop the same stored sequence to:

- 51 bp
- 101 bp
- 201 bp

without having to access the reference genome again.

The centre base is checked against the ClinVar REF allele as a coordinate / assembly sanity test.

In [18]:
import py2bit
import pyBigWig

genome = py2bit.open(str(LOCAL_2BIT))
bw_phylop = pyBigWig.open(str(LOCAL_PHYLOP))
bw_phastcons = pyBigWig.open(str(LOCAL_PHASTCONS))


def ucsc_chrom(chrom):
    chrom = str(chrom).replace('chr', '')
    return f"chr{chrom}"


def extract_seq201(chrom, pos_1based):
    # GRCh38 coordinates in variant_summary are 1-based.
    # py2bit uses 0-based half-open coordinates.

    chrom = ucsc_chrom(chrom)
    center0 = int(pos_1based) - 1

    start0 = center0 - 100
    end0 = center0 + 101

    if start0 < 0:
        return None

    try:
        seq = genome.sequence(
            chrom,
            start0,
            end0
        )
    except Exception:
        return None

    if seq is None:
        return None

    seq = seq.upper()

    if len(seq) != 201:
        return None

    return seq


def bigwig_value(bw, chrom, pos_1based):
    chrom = ucsc_chrom(chrom)
    p0 = int(pos_1based) - 1

    try:
        values = bw.values(
            chrom,
            p0,
            p0 + 1
        )

        if not values:
            return np.nan

        value = values[0]

        if value is None:
            return np.nan

        return float(value)

    except Exception:
        return np.nan

In [19]:
seq201_list = []
phylop_list = []
phastcons_list = []
ref_match_list = []

for row in tqdm(
    sampled_df.itertuples(index=False),
    total=len(sampled_df),
    desc="GRCh38 + conservation annotation"
):

    seq201 = extract_seq201(
        row.Chromosome,
        row.Start
    )

    seq201_list.append(seq201)

    phylop_list.append(
        bigwig_value(
            bw_phylop,
            row.Chromosome,
            row.Start
        )
    )

    phastcons_list.append(
        bigwig_value(
            bw_phastcons,
            row.Chromosome,
            row.Start
        )
    )

    if seq201 is None:
        ref_match_list.append(False)

    else:
        ref_match_list.append(
            seq201[100] == row.ReferenceAlleleVCF
        )


sampled_df['seq201'] = seq201_list
sampled_df['phylop'] = np.asarray(
    phylop_list,
    dtype=np.float32
)

sampled_df['phastcons'] = np.asarray(
    phastcons_list,
    dtype=np.float32
)

sampled_df['ref_match'] = ref_match_list


print("201 bp window availability:",
      f"{sampled_df['seq201'].notna().mean():.4%}")

print("REF match rate:",
      f"{sampled_df['ref_match'].mean():.4%}")

print("Missing PhyloP:",
      f"{sampled_df['phylop'].isna().mean():.4%}")

print("Missing PhastCons:",
      f"{sampled_df['phastcons'].isna().mean():.4%}")

GRCh38 + conservation annotation:   0%|          | 0/100000 [00:00<?, ?it/s]

201 bp window availability: 100.0000%
REF match rate: 100.0000%
Missing PhyloP: 0.0040%
Missing PhastCons: 0.0040%


### REF-match sanity check

Do **not** ignore a poor REF-match rate.

A large number of mismatches usually means:

- wrong reference assembly,
- incorrect coordinate convention,
- malformed variants,
- or bad input rows.

Only rows with a valid 201 bp window and a matching GRCh38 REF allele continue to the final feature table.

In [20]:
bad = sampled_df[
    (~sampled_df['ref_match']) |
    (sampled_df['seq201'].isna())
]

print("Invalid / REF-mismatch rows:", len(bad))

display(
    bad[
        [
            'Chromosome',
            'Start',
            'ReferenceAlleleVCF',
            'AlternateAlleleVCF',
            'GeneSymbol',
            'seq201'
        ]
    ].head(20)
)

sampled_df = sampled_df[
    sampled_df['ref_match'] &
    sampled_df['seq201'].notna()
].reset_index(drop=True)

# Reassign IDs after filtering so graph node IDs are contiguous.
sampled_df['node_id'] = np.arange(
    len(sampled_df),
    dtype=np.int64
)

print("Rows remaining:", f"{len(sampled_df):,}")

Invalid / REF-mismatch rows: 0


,Chromosome,Start,ReferenceAlleleVCF,AlternateAlleleVCF,GeneSymbol,seq201


Rows remaining: 100,000


# Step 2: Add Molecular Consequence

The original `variant_summary.txt` pipeline does not provide the detailed molecular-consequence field needed by the upgraded model.

To preserve the original notebook's data source while adding this missing component, this section streams the official **ClinVar GRCh38 VCF** once and keeps `MC` only for variants that are already in our sampled panel.

This is much lighter than loading the entire VCF into memory.

In [21]:
# Build a lookup only for the variants in sampled_df.

sample_keys = {
    (
        str(chrom).replace('chr', ''),
        int(pos),
        str(ref).upper(),
        str(alt).upper()
    )
    for chrom, pos, ref, alt in zip(
        sampled_df['Chromosome'],
        sampled_df['Start'],
        sampled_df['ReferenceAlleleVCF'],
        sampled_df['AlternateAlleleVCF']
    )
}

print("Keys to match:", f"{len(sample_keys):,}")

Keys to match: 100,000


In [22]:
def parse_info_field(info):
    out = {}

    for token in info.split(';'):

        if '=' in token:
            key, value = token.split('=', 1)
            out[key] = value

    return out


mc_lookup = {}

with gzip.open(CLINVAR_VCF, 'rt') as f:

    for line in tqdm(
        f,
        desc="Matching ClinVar molecular consequences"
    ):

        if line.startswith('#'):
            continue

        fields = line.rstrip('\n').split('\t')

        if len(fields) < 8:
            continue

        chrom, pos, _vid, ref, alt, _qual, _filter, info = fields[:8]

        chrom = chrom.replace('chr', '')

        try:
            pos = int(pos)
        except Exception:
            continue

        info_dict = None

        # Split a multi-allelic site into separate candidate keys.
        for alt_allele in alt.split(','):

            key = (
                chrom,
                pos,
                ref.upper(),
                alt_allele.upper()
            )

            if key not in sample_keys:
                continue

            if info_dict is None:
                info_dict = parse_info_field(info)

            mc_lookup[key] = info_dict.get(
                'MC',
                ''
            )

print("Matched consequences:", f"{len(mc_lookup):,}")

Matching ClinVar molecular consequences: 0it [00:00, ?it/s]

Matched consequences: 99,936


In [23]:
sampled_df['molecular_consequence_raw'] = [
    mc_lookup.get(
        (
            str(chrom).replace('chr', ''),
            int(pos),
            str(ref).upper(),
            str(alt).upper()
        ),
        ''
    )
    for chrom, pos, ref, alt in zip(
        sampled_df['Chromosome'],
        sampled_df['Start'],
        sampled_df['ReferenceAlleleVCF'],
        sampled_df['AlternateAlleleVCF']
    )
]


def consequence_category(mc):
    text = str(mc).lower()

    if 'missense_variant' in text:
        return 'missense'

    if (
        'stop_gained' in text or
        'stop_lost' in text or
        'start_lost' in text
    ):
        return 'nonsense_or_stop'

    if 'synonymous_variant' in text:
        return 'synonymous'

    if 'splice' in text:
        return 'splice'

    if 'intron_variant' in text:
        return 'intron'

    if (
        '5_prime_utr_variant' in text or
        '3_prime_utr_variant' in text
    ):
        return 'utr'

    if (
        'upstream_gene_variant' in text or
        'downstream_gene_variant' in text
    ):
        return 'near_gene'

    if text.strip() == '':
        return 'unknown'

    return 'other'


sampled_df['consequence_category'] = (
    sampled_df['molecular_consequence_raw']
    .apply(consequence_category)
)

print(
    sampled_df['consequence_category']
    .value_counts(dropna=False)
)

consequence_category
missense            62645
synonymous          18904
intron              13087
other                1981
utr                  1716
splice               1303
unknown               330
nonsense_or_stop       34
Name: count, dtype: int64


# Step 3: Population-Frequency Schema

## gnomAD note for free Colab

Bulk gnomAD annotation is deliberately **not downloaded as a whole-genome VCF in this notebook**.

Why:

- the full gnomAD site resources are very large,
- random lookup across 100k–300k variants is not a good free-Colab workflow,
- and the project should not silently mix genome assemblies or unreliable coordinate conversions.

The final schema still includes:

- `gnomad_af`
- `gnomad_present`

They are initially missing/zero here.

A later cached gnomAD annotation table can be merged by the exact GRCh38 key:

`Chromosome + Start + REF + ALT`

This lets Notebook 2 run now while preserving a clean place for the final population-frequency feature.

For **live VCF inference**, gnomAD lookups will be performed only for the small number of uploaded variants and cached by the backend.

In [24]:
# Stable schema for Notebook 2.
# These columns can later be filled by merging a verified GRCh38 gnomAD cache.

if 'gnomad_af' not in sampled_df.columns:
    sampled_df['gnomad_af'] = np.nan

if 'gnomad_present' not in sampled_df.columns:
    sampled_df['gnomad_present'] = np.int8(0)

print(
    sampled_df[
        ['gnomad_af', 'gnomad_present']
    ].head()
)

   gnomad_af  gnomad_present
0        NaN               0
1        NaN               0
2        NaN               0
3        NaN               0
4        NaN               0


# Step 4: Final Feature Table

The final file contains raw values as much as possible.

Notebook 2 will:

- crop `seq201` to 101 bp,
- build the 4 × 101 sequence tensor,
- one-hot encode categorical metadata,
- fit scaling on **training folds only**,
- construct the graph,
- train CNN + MLP + GCN.

Keeping preprocessing and model fitting separate prevents accidental leakage.

In [25]:
# Ensure the key columns have stable dtypes.
sampled_df['Chromosome'] = sampled_df['Chromosome'].astype(str)
sampled_df['Start'] = sampled_df['Start'].astype(np.int64)
sampled_df['Label'] = sampled_df['Label'].astype(np.int8)
sampled_df['node_id'] = sampled_df['node_id'].astype(np.int64)

# A compact exact variant key is useful for external-validation overlap checks.
sampled_df['variant_key'] = (
    sampled_df['Chromosome'].astype(str)
    + ':'
    + sampled_df['Start'].astype(str)
    + ':'
    + sampled_df['ReferenceAlleleVCF'].astype(str)
    + '>'
    + sampled_df['AlternateAlleleVCF'].astype(str)
)

final_columns = [
    # Identity / coordinates
    'node_id',
    'variant_key',
    'Chromosome',
    'Start',
    'Stop',
    'ReferenceAlleleVCF',
    'AlternateAlleleVCF',
    'GeneSymbol',

    # Target / raw ClinVar metadata
    'Type',
    'ClinicalSignificance',
    'Label',
    'Assembly',
    'OriginSimple',
    'ReviewStatus',
    'NumberSubmitters',
    'SomaticClinicalImpact',

    # Legacy prototype features kept for reproducibility
    'Ref_A', 'Ref_C', 'Ref_G', 'Ref_T',
    'Alt_A', 'Alt_C', 'Alt_G', 'Alt_T',
    'Tab_Scaled_Submitters',
    'Tab_Origin_Numeric',
    'Tab_Review_Score',
    'Tab_Somatic_Impact',

    # Upgraded biological features
    'seq201',
    'phylop',
    'phastcons',
    'molecular_consequence_raw',
    'consequence_category',
    'gnomad_af',
    'gnomad_present',

    # Data validation
    'ref_match'
]

final_columns = [
    c for c in final_columns
    if c in sampled_df.columns
]

final_df = sampled_df[
    final_columns
].copy()

print("Final rows:", f"{len(final_df):,}")
print("Final columns:", len(final_df.columns))

display(final_df.head())

Final rows: 100,000
Final columns: 36


,node_id,variant_key,Chromosome,Start,Stop,ReferenceAlleleVCF,AlternateAlleleVCF,GeneSymbol,Type,ClinicalSignificance,...,Tab_Review_Score,Tab_Somatic_Impact,seq201,phylop,phastcons,molecular_consequence_raw,consequence_category,gnomad_af,gnomad_present,ref_match
0,0,7:107917471:G>A,7,107917471,107917471,G,A,DLD,single nucleotide variant,Likely benign,...,1.0,1.0,TGCACATTGACTACAATTGTGTGCCATCAGTGATTTACACACACCC...,0.579,0.013,SO:0001627|intron_variant,intron,NaN,0,True
1,1,17:76024068:C>T,17,76024068,76024068,C,T,EVPL,single nucleotide variant,Uncertain significance,...,1.0,1.0,CTCCCATGCCACCCAGCCTGTCCACGCCCTGCCAACTGCTGCCGGG...,7.350,1.000,SO:0001583|missense_variant,missense,NaN,0,True
2,2,2:165873829:G>T,2,165873829,165873829,G,T,TTC21B,single nucleotide variant,Uncertain significance,...,1.0,1.0,TATTTCTTAAGACCACAATGGCCTTGAAAGCAGGAAGAATGAGCTC...,1.249,0.929,SO:0001624|3_prime_UTR_variant,utr,NaN,0,True
3,3,17:42900997:A>G,17,42900997,42900997,A,G,G6PC1,single nucleotide variant,Uncertain significance,...,1.0,1.0,TCTCCATGACTTTGGGATCCAGTCAACACATTACCTCCAGGTGAAT...,9.236,1.000,SO:0001583|missense_variant,missense,NaN,0,True
4,4,10:5960432:C>G,10,5960432,5960432,C,G,IL15RA,single nucleotide variant,Likely benign,...,1.0,1.0,GAACTGACCTCTCCTGGGGCAAAGCGAGTGCTAACCTGGCGGCTGG...,-0.409,0.000,"SO:0001583|missense_variant,SO:0001619|non-cod...",missense,NaN,0,True


## Final sanity checks

In [26]:
print("================ FINAL DATA CHECKS ================")

print("\nRows:")
print(f"{len(final_df):,}")

print("\nClass distribution:")
print(final_df['Label'].value_counts().sort_index())

print("\nClass percentages:")
print(
    (
        final_df['Label']
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)

print("\nDuplicate variant keys:")
print(
    final_df['variant_key']
    .duplicated()
    .sum()
)

print("\nREF match:")
print(
    final_df['ref_match']
    .value_counts(dropna=False)
)

print("\nSequence lengths:")
print(
    final_df['seq201']
    .dropna()
    .str.len()
    .value_counts()
    .head()
)

print("\nMissing values in upgraded features:")
for col in [
    'seq201',
    'phylop',
    'phastcons',
    'molecular_consequence_raw',
    'consequence_category',
    'gnomad_af'
]:
    if col in final_df.columns:
        print(
            f"{col:28s}",
            f"{final_df[col].isna().mean():.2%}"
        )

assert final_df['ref_match'].all()
assert final_df['seq201'].str.len().eq(201).all()
assert final_df['node_id'].is_unique

print("\n✅ Core Notebook-1 checks passed.")

================ FINAL DATA CHECKS ================

Rows:
100,000

Class distribution:
Label
0    34621
1     5135
2    60244
Name: count, dtype: int64

Class percentages:
Label
0    34.62
1     5.14
2    60.24
Name: proportion, dtype: float64

Duplicate variant keys:
0

REF match:
ref_match
True    100000
Name: count, dtype: int64

Sequence lengths:
seq201
201    100000
Name: count, dtype: int64

Missing values in upgraded features:
seq201                       0.00%
phylop                       0.00%
phastcons                    0.00%
molecular_consequence_raw    0.00%
consequence_category         0.00%
gnomad_af                    100.00%

✅ Core Notebook-1 checks passed.


## Save final dataset

This is the **only file Notebook 2 needs to load**.

In [27]:
FINAL_OUTPUT = (
    PROCESSED_DIR /
    f"genohybrid_features_{len(final_df)}.parquet"
)

final_df.to_parquet(
    FINAL_OUTPUT,
    index=False
)

# Also write a stable filename so Notebook 2 does not need
# to know the sample size.
STABLE_OUTPUT = (
    PROCESSED_DIR /
    "genohybrid_features.parquet"
)

final_df.to_parquet(
    STABLE_OUTPUT,
    index=False
)

print("Saved:")
print(FINAL_OUTPUT)
print(STABLE_OUTPUT)

print(
    "File size:",
    f"{STABLE_OUTPUT.stat().st_size / 1e6:.1f} MB"
)

Saved:
/content/drive/MyDrive/variant_effect_prediction/data/processed/genohybrid_features_100000.parquet
/content/drive/MyDrive/variant_effect_prediction/data/processed/genohybrid_features.parquet
File size: 14.2 MB


## Save a run manifest

This records the preprocessing configuration used to create the file.

In [28]:
manifest = {
    "sample_size_requested": SAMPLE_SIZE,
    "sample_size_saved": int(len(final_df)),
    "random_seed": SEED,
    "reference_assembly": "GRCh38",
    "sequence_storage_window_bp": 201,
    "target_classes": {
        "0": "Benign",
        "1": "Pathogenic",
        "2": "VUS"
    },
    "target_leakage_feature_removed": "Tab_Gene_Vulnerability",
    "gnomad_status": (
        "schema reserved; bulk population-frequency annotation "
        "intentionally separated from free-Colab preprocessing"
    ),
    "output": str(STABLE_OUTPUT)
}

MANIFEST_PATH = (
    PROCESSED_DIR /
    "genohybrid_features_manifest.json"
)

with open(
    MANIFEST_PATH,
    "w"
) as f:
    json.dump(
        manifest,
        f,
        indent=2
    )

print(json.dumps(manifest, indent=2))

{
  "sample_size_requested": 100000,
  "sample_size_saved": 100000,
  "random_seed": 42,
  "reference_assembly": "GRCh38",
  "sequence_storage_window_bp": 201,
  "target_classes": {
    "0": "Benign",
    "1": "Pathogenic",
    "2": "VUS"
  },
  "target_leakage_feature_removed": "Tab_Gene_Vulnerability",
  "gnomad_status": "schema reserved; bulk population-frequency annotation intentionally separated from free-Colab preprocessing",
  "output": "/content/drive/MyDrive/variant_effect_prediction/data/processed/genohybrid_features.parquet"
}


In [30]:
len(final_df)

100000

# Notebook 1 Complete

### Notebook 2 will handle

1. gene-grouped split / GroupKFold,
2. 101 bp sequence tensor generation from `seq201`,
3. leakage-free scaling and categorical encoding,
4. CNN sequence arm,
5. MLP biomarker arm,
6. 2-hop batched GCN arm,
7. late fusion,
8. class-weighted training,
9. Macro-F1 / MCC / AUROC / confusion matrix,
10. Monte Carlo dropout,
11. checkpoint saving.